# Turkish streaming RNN-T training

Türkçe-özel, düşük gecikmeli causal Conformer-RNN-T eğitimi. Bu notebook veri indirmez. Mozilla Common Voice Türkçe verisini elle `data/common_voice_tr/` altına yerleştirdikten sonra çalıştırın.

Beklenen yapı:
```text
data/common_voice_tr/train.tsv
data/common_voice_tr/dev.tsv
data/common_voice_tr/clips/*.mp3
```

In [ ]:
# Hyperparameters: autoresearch bu hücreyi değiştirebilir.
from __future__ import annotations

import csv
import itertools
import random
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import torch
import torchaudio
from torch import nn
from torch.utils.data import DataLoader, Dataset

COMMON_VOICE_ROOT = Path('data/common_voice_tr')
SEED = 1337
BATCH_SIZE = 8
NUM_WORKERS = 4
MIN_AUDIO_SECONDS = 0.35
MAX_AUDIO_SECONDS = 15.0
ENCODER_DIM = 256
ENCODER_LAYERS = 8
ATTENTION_HEADS = 4
CONV_KERNEL = 15
PREDICTOR_DIM = 192
JOINER_DIM = 256
DROPOUT = 0.1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
ACCUMULATION_STEPS = 2
MAX_STEPS = 100_000
VALIDATE_EVERY = 1_000
SAVE_EVERY = 1_000
GRAD_CLIP_NORM = 5.0
CHECKPOINT_DIR = Path('checkpoints')

In [ ]:
class TurkishCharTokenizer:
    # Grapheme tokenizer: Turkish morphology and names have no OOV words.
    blank_id, bos_id, pad_id = 0, 1, 2
    symbols = ' abcçdefgğhıijklmnoöprsştuüvyz\'.,?!'

    def __init__(self):
        self.id_to_token = ['<blank>', '<bos>', '<pad>', *self.symbols]
        self.token_to_id = {token: index for index, token in enumerate(self.id_to_token)}

    @property
    def vocab_size(self):
        return len(self.id_to_token)

    def normalize(self, text):
        text = unicodedata.normalize('NFC', text).lower()
        text = re.sub(r"[^ abcçdefgğhıijklmnoöprsştuüvyz'.,?!]", ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    def encode(self, text):
        return [self.token_to_id[char] for char in self.normalize(text)]

tokenizer = TurkishCharTokenizer()
print('vocab size:', tokenizer.vocab_size)

In [ ]:
@dataclass(frozen=True)
class Sample:
    audio_path: Path
    sentence: str

class CommonVoiceTurkish(Dataset):
    def __init__(self, tsv_name):
        manifest = COMMON_VOICE_ROOT / tsv_name
        if not manifest.is_file():
            raise FileNotFoundError(f'{manifest} missing. This notebook never downloads data.')
        self.samples = []
        with manifest.open(encoding='utf-8', newline='') as handle:
            for row in csv.DictReader(handle, delimiter='\t'):
                sentence = tokenizer.normalize(row.get('sentence', ''))
                path = COMMON_VOICE_ROOT / 'clips' / row.get('path', '')
                if sentence and path.is_file():
                    self.samples.append(Sample(path, sentence))
        if not self.samples:
            raise RuntimeError(f'No readable samples in {manifest}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        waveform, rate = torchaudio.load(sample.audio_path)
        waveform = waveform.mean(dim=0, keepdim=True)
        if rate != 16_000:
            waveform = torchaudio.functional.resample(waveform, rate, 16_000)
        seconds = waveform.size(1) / 16_000
        if not MIN_AUDIO_SECONDS <= seconds <= MAX_AUDIO_SECONDS:
            raise RuntimeError(f'Out-of-range duration: {sample.audio_path}')
        features = torchaudio.compliance.kaldi.fbank(waveform, sample_frequency=16_000, num_mel_bins=80, dither=0.0)
        return features, torch.tensor(tokenizer.encode(sample.sentence), dtype=torch.long)

def collate(items):
    features, targets = zip(*items)
    return {
        'features': nn.utils.rnn.pad_sequence(features, batch_first=True),
        'feature_lengths': torch.tensor([x.size(0) for x in features]),
        'targets': nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=tokenizer.pad_id),
        'target_lengths': torch.tensor([x.size(0) for x in targets]),
    }

def loader(tsv_name, shuffle):
    return DataLoader(CommonVoiceTurkish(tsv_name), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, collate_fn=collate)

In [ ]:
class CausalConvSubsampler(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.layers = nn.ModuleList([nn.Conv1d(input_dim, output_dim, 5, stride=2), nn.Conv1d(output_dim, output_dim, 5, stride=2)])

    def forward(self, x, lengths):
        x = x.transpose(1, 2)
        for layer in self.layers:
            x = torch.nn.functional.gelu(layer(torch.nn.functional.pad(x, (4, 0))))
            lengths = (lengths + 1) // 2
        return x.transpose(1, 2), lengths

class CausalConformerBlock(nn.Module):
    def __init__(self, dim, heads, kernel, dropout):
        super().__init__()
        self.ff1 = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim * 4), nn.SiLU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))
        self.attn_norm, self.attn = nn.LayerNorm(dim), nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.conv_norm = nn.LayerNorm(dim)
        self.pw_in, self.dw, self.pw_out = nn.Conv1d(dim, dim * 2, 1), nn.Conv1d(dim, dim, kernel, groups=dim), nn.Conv1d(dim, dim, 1)
        self.ff2 = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim * 4), nn.SiLU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))
        self.final_norm, self.dropout, self.kernel = nn.LayerNorm(dim), nn.Dropout(dropout), kernel

    def forward(self, x, lengths):
        x = x + 0.5 * self.dropout(self.ff1(x))
        steps = x.size(1)
        causal = torch.ones(steps, steps, device=x.device, dtype=torch.bool).triu(1)
        padding = torch.arange(steps, device=x.device)[None, :] >= lengths[:, None]
        q = self.attn_norm(x)
        attended, _ = self.attn(q, q, q, attn_mask=causal, key_padding_mask=padding, need_weights=False)
        x = x + self.dropout(attended)
        conv = self.conv_norm(x).transpose(1, 2)
        left, right = self.pw_in(conv).chunk(2, dim=1)
        conv = torch.nn.functional.silu(self.dw(torch.nn.functional.pad(left * torch.sigmoid(right), (self.kernel - 1, 0))))
        x = x + self.dropout(self.pw_out(conv).transpose(1, 2))
        return self.final_norm(x + 0.5 * self.dropout(self.ff2(x)))

class StreamingTransducer(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.subsample = CausalConvSubsampler(80, ENCODER_DIM)
        self.encoder = nn.ModuleList([CausalConformerBlock(ENCODER_DIM, ATTENTION_HEADS, CONV_KERNEL, DROPOUT) for _ in range(ENCODER_LAYERS)])
        self.embedding = nn.Embedding(vocab_size, PREDICTOR_DIM)
        self.predictor = nn.GRU(PREDICTOR_DIM, PREDICTOR_DIM, batch_first=True)
        self.encoder_proj, self.predictor_proj = nn.Linear(ENCODER_DIM, JOINER_DIM), nn.Linear(PREDICTOR_DIM, JOINER_DIM)
        self.joiner = nn.Sequential(nn.Tanh(), nn.Linear(JOINER_DIM, vocab_size))

    def forward(self, features, feature_lengths, targets):
        encoded, lengths = self.subsample(features, feature_lengths)
        for block in self.encoder: encoded = block(encoded, lengths)
        bos = torch.full((targets.size(0), 1), tokenizer.bos_id, device=targets.device, dtype=targets.dtype)
        predicted, _ = self.predictor(self.embedding(torch.cat((bos, targets), dim=1)))
        logits = self.joiner(self.encoder_proj(encoded).unsqueeze(2) + self.predictor_proj(predicted).unsqueeze(1))
        return logits, lengths

In [ ]:
# Training. Run only after placing Common Voice locally; no download occurs here.
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader, valid_loader = loader('train.tsv', True), loader('dev.tsv', False)
model = StreamingTransducer(tokenizer.vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

def move(batch):
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}

optimizer.zero_grad(set_to_none=True)
train_iter = itertools.cycle(train_loader)
for step in range(1, MAX_STEPS + 1):
    model.train()
    batch = move(next(train_iter))
    with torch.autocast(device_type=device.type, enabled=scaler.is_enabled()):
        logits, lengths = model(batch['features'], batch['feature_lengths'], batch['targets'])
        loss = torchaudio.functional.rnnt_loss(logits, batch['targets'], lengths, batch['target_lengths'], blank=tokenizer.blank_id, reduction='mean', fused_log_softmax=True) / ACCUMULATION_STEPS
    scaler.scale(loss).backward()
    if step % ACCUMULATION_STEPS == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    if step % 50 == 0:
        print(f'step={step} train_rnnt_loss={loss.item() * ACCUMULATION_STEPS:.4f}')
    if step % VALIDATE_EVERY == 0:
        model.eval()
        with torch.inference_mode():
            valid = move(next(iter(valid_loader)))
            valid_logits, valid_lengths = model(valid['features'], valid['feature_lengths'], valid['targets'])
            valid_loss = torchaudio.functional.rnnt_loss(valid_logits, valid['targets'], valid_lengths, valid['target_lengths'], blank=tokenizer.blank_id, reduction='mean', fused_log_softmax=True)
        print(f'step={step} valid_rnnt_loss={valid_loss.item():.4f}')
    if step % SAVE_EVERY == 0:
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        torch.save({'step': step, 'model': model.state_dict(), 'optimizer': optimizer.state_dict()}, CHECKPOINT_DIR / f'step-{step}.pt')